# Phase 3: Emotion-Conditioned TTS Fine-Tuning
**Owner: Member A**

This phase trains **first**. It exports the emotion lookup table for Phase 2.

All model logic lives in `emonarrify/phase3/model.py`.

In [6]:
# ===== Setup =====
# Local:
#   python -m pip install -r requirements.txt
# Colab:
# !git clone https://github.com/YOUR_REPO/emonarrify.git
# %cd emonarrify
# !pip install -r requirements.txt

import os
import sys
import json
import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, os.path.abspath('..'))

from emonarrify.config import D_EMO, EMOTION_LABELS, LOOKUP_TABLE_PATH, SAMPLE_RATE
from emonarrify.phase3.model import EmotionLookupTable, ConditionalNeuralTTSBackbone

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_ROOT = Path('..').resolve()
WEIGHTS_DIR = PROJECT_ROOT / 'weights'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
DATA_DIR = PROJECT_ROOT / 'data'

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'D_emo: {D_EMO}, Emotions: {EMOTION_LABELS}, SR: {SAMPLE_RATE}')

Device: cpu
D_emo: 128, Emotions: ['neutral', 'happy', 'angry', 'sad', 'surprise'], SR: 22050


In [ ]:
# ===== Colab Helper (commented) =====
# target folder structure: ./ESD/<speaker_id>/<Emotion>/*.wav

# !pip install -q gdown
# import os
#
# # Google Drive file id from
# # https://drive.google.com/file/d/1scuFwqh8s7KIYAfZW1Eu6088ZAK2SI-v/view?usp=sharing
# FILE_ID = "1scuFwqh8s7KIYAfZW1Eu6088ZAK2SI-v"
# ZIP_PATH = "ESD.zip"
#
# !gdown --id $FILE_ID -O $ZIP_PATH
# !unzip -q $ZIP_PATH -d .
#
# # if the extracted folder name is not "ESD", rename it to "ESD"
# # example: !mv extracted_folder_name ESD
#
# assert os.path.isdir("ESD"), "ESD folder not found after unzip"

In [7]:
# ===== Data: Load ESD via manifest =====
# We train with records generated by: data/phase3_prepare_data.py

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'phase3_esd_manifest.jsonl'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Manifest not found: {MANIFEST_PATH}\n'
        'Please run: python data/phase3_prepare_data.py --esd-root ESD --output data/phase3_esd_manifest.jsonl'
    )

with MANIFEST_PATH.open('r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f if line.strip()]

train_records = [r for r in records if str(r.get('split', '')).lower() == 'train']
val_records = [r for r in records if str(r.get('split', '')).lower() == 'val']

def _emotion_to_idx(label: str) -> int:
    return EMOTION_LABELS.index(str(label).strip().lower())


def _resample_waveform_np(wav: np.ndarray, src_sr: int, tgt_sr: int) -> np.ndarray:
    """Simple linear-resample fallback without torchaudio dependency."""
    if src_sr == tgt_sr:
        return wav.astype(np.float32)
    wav_t = torch.tensor(wav, dtype=torch.float32)
    if wav_t.dim() == 1:
        wav_t = wav_t.unsqueeze(0).unsqueeze(0)
    else:
        wav_t = wav_t.mean(dim=-1, keepdim=False).unsqueeze(0).unsqueeze(0)
    new_len = max(1, int(round(wav_t.shape[-1] * tgt_sr / src_sr)))
    out = F.interpolate(wav_t, size=new_len, mode='linear', align_corners=False)
    return out.squeeze().cpu().numpy().astype(np.float32)


class ESDPhase3Dataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rec = self.items[idx]
        text = rec['narrative_text']
        emotion_label = str(rec['emotion_label']).strip().lower()
        emotion_idx = _emotion_to_idx(emotion_label)

        audio_path = Path(rec['audio_path'])
        wav, sr = sf.read(str(audio_path), dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = _resample_waveform_np(wav, sr, SAMPLE_RATE)

        return {
            'text': text,
            'emotion_label': emotion_label,
            'emotion_idx': emotion_idx,
            'gt_wav': torch.tensor(wav, dtype=torch.float32),
            'audio_path': str(audio_path),
        }


def collate_fn(batch):
    return batch

BATCH_SIZE = 8
train_loader = DataLoader(ESDPhase3Dataset(train_records), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ESDPhase3Dataset(val_records), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Total records: {len(records)} | train: {len(train_records)} | val: {len(val_records)}')

Total records: 35000 | train: 28000 | val: 3500


In [8]:
# ===== Model Setup =====

lookup = EmotionLookupTable().to(DEVICE)
tts_backbone = ConditionalNeuralTTSBackbone(sample_rate=SAMPLE_RATE).to(DEVICE)

# Jointly optimize TTS + lookup table
optimizer = torch.optim.AdamW(
    list(tts_backbone.parameters()) + list(lookup.parameters()),
    lr=2e-4,
    weight_decay=1e-5,
)

# Optional: cosine LR scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=5e-6)

print('Model ready')
print(f'TTS params: {sum(p.numel() for p in tts_backbone.parameters()):,}')
print(f'Lookup params: {sum(p.numel() for p in lookup.parameters()):,}')

Model ready
TTS params: 1,064,652
Lookup params: 640


In [9]:
# ===== Training =====

def align_pred_to_target(pred_wav: torch.Tensor, target_wav: torch.Tensor) -> torch.Tensor:
    """Align predicted waveform length to target waveform length."""
    if pred_wav.dim() == 1:
        pred_wav = pred_wav.unsqueeze(0).unsqueeze(0)
    else:
        pred_wav = pred_wav.unsqueeze(0)
    target_len = target_wav.shape[-1]
    pred_aligned = F.interpolate(pred_wav, size=target_len, mode='linear', align_corners=False)
    return pred_aligned.squeeze(0).squeeze(0)


def stft_mag_loss(pred: torch.Tensor, target: torch.Tensor, n_fft: int = 512, hop_length: int = 128) -> torch.Tensor:
    window = torch.hann_window(n_fft, device=pred.device)
    pred_stft = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, win_length=n_fft, window=window, return_complex=True)
    tgt_stft = torch.stft(target, n_fft=n_fft, hop_length=hop_length, win_length=n_fft, window=window, return_complex=True)
    return F.l1_loss(torch.abs(pred_stft), torch.abs(tgt_stft))


def batch_forward_and_loss(batch):
    total_loss = 0.0
    details = []

    for sample in batch:
        text = sample['text']
        gt = sample['gt_wav'].to(DEVICE)
        emo_idx = torch.tensor(sample['emotion_idx'], device=DEVICE, dtype=torch.long)

        emo_emb = lookup(emo_idx)
        pred = tts_backbone.synthesize(text=text, emotion_embedding=emo_emb)
        pred = align_pred_to_target(pred, gt)

        # Time-domain + spectral loss
        loss_l1 = F.l1_loss(pred, gt)
        loss_spec = stft_mag_loss(pred, gt)
        loss = loss_l1 + 0.5 * loss_spec

        total_loss = total_loss + loss
        details.append((loss_l1.item(), loss_spec.item()))

    total_loss = total_loss / max(1, len(batch))
    return total_loss, details


EPOCHS = 1
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    tts_backbone.train()
    lookup.train()

    train_losses = []
    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad(set_to_none=True)
        loss, details = batch_forward_and_loss(batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(tts_backbone.parameters()) + list(lookup.parameters()), max_norm=1.0)
        optimizer.step()

        train_losses.append(loss.item())

        if step % 50 == 0:
            print(f'[Epoch {epoch}] step {step}/{len(train_loader)} | train_loss={np.mean(train_losses[-50:]):.4f}')

    # ---- Validation ----
    tts_backbone.eval()
    lookup.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            val_loss, _ = batch_forward_and_loss(batch)
            val_losses.append(val_loss.item())

    epoch_train = float(np.mean(train_losses)) if train_losses else float('nan')
    epoch_val = float(np.mean(val_losses)) if val_losses else float('nan')
    print(f'\n[Epoch {epoch}] train={epoch_train:.4f} | val={epoch_val:.4f}')

    if not math.isnan(epoch_val) and epoch_val < best_val:
        best_val = epoch_val
        ckpt_path = WEIGHTS_DIR / 'phase3_tts.pt'
        torch.save(
            {
                'tts_model': tts_backbone.state_dict(),
                'lookup_table': lookup.state_dict(),
                'epoch': epoch,
                'val_loss': epoch_val,
                'sample_rate': SAMPLE_RATE,
                'model_name': 'ConditionalNeuralTTSBackbone',
            },
            ckpt_path,
        )
        print(f'  -> saved best checkpoint to: {ckpt_path}')

    scheduler.step()

[Epoch 1] step 50/3500 | train_loss=0.9570
[Epoch 1] step 100/3500 | train_loss=0.7247
[Epoch 1] step 150/3500 | train_loss=0.6606
[Epoch 1] step 200/3500 | train_loss=0.6018
[Epoch 1] step 250/3500 | train_loss=0.5938
[Epoch 1] step 300/3500 | train_loss=0.5768
[Epoch 1] step 350/3500 | train_loss=0.5846
[Epoch 1] step 400/3500 | train_loss=0.5672
[Epoch 1] step 450/3500 | train_loss=0.5669
[Epoch 1] step 500/3500 | train_loss=0.5659
[Epoch 1] step 550/3500 | train_loss=0.5652
[Epoch 1] step 600/3500 | train_loss=0.5632
[Epoch 1] step 650/3500 | train_loss=0.5683
[Epoch 1] step 700/3500 | train_loss=0.5688
[Epoch 1] step 750/3500 | train_loss=0.5630
[Epoch 1] step 800/3500 | train_loss=0.5539
[Epoch 1] step 850/3500 | train_loss=0.5647
[Epoch 1] step 900/3500 | train_loss=0.5672
[Epoch 1] step 950/3500 | train_loss=0.5720
[Epoch 1] step 1000/3500 | train_loss=0.5635
[Epoch 1] step 1050/3500 | train_loss=0.5649
[Epoch 1] step 1100/3500 | train_loss=0.5635
[Epoch 1] step 1150/3500 | tra

In [10]:
# ===== Validation =====
# 1) Print lookup cosine similarity matrix
# 2) Synthesize one sentence with all 5 emotions for quick A/B listening

lookup.eval()
tts_backbone.eval()

sim = lookup.cosine_similarity_matrix()
print('Cosine similarity matrix:')
print(np.array2string(sim, precision=3))

val_text = 'A quiet scene unfolds under a pale sky, carrying a sense of calm solitude.'
for emo in EMOTION_LABELS:
    emo_idx = torch.tensor(EMOTION_LABELS.index(emo), device=DEVICE, dtype=torch.long)
    with torch.no_grad():
        emo_emb = lookup(emo_idx)
        wav = tts_backbone.synthesize(val_text, emo_emb).detach().cpu().numpy().astype(np.float32)

    out_path = OUTPUTS_DIR / f'phase3_val_{emo}.wav'
    sf.write(str(out_path), wav, SAMPLE_RATE)
    print(f'Saved: {out_path}')

Cosine similarity matrix:
[[ 1.    -0.015  0.13  -0.09  -0.083]
 [-0.015  1.    -0.031 -0.005  0.016]
 [ 0.13  -0.031  1.    -0.077 -0.131]
 [-0.09  -0.005 -0.077  1.     0.004]
 [-0.083  0.016 -0.131  0.004  1.   ]]
Saved: /Users/andrewchen/Desktop/18789/emonarrify/outputs/phase3_val_neutral.wav
Saved: /Users/andrewchen/Desktop/18789/emonarrify/outputs/phase3_val_happy.wav
Saved: /Users/andrewchen/Desktop/18789/emonarrify/outputs/phase3_val_angry.wav
Saved: /Users/andrewchen/Desktop/18789/emonarrify/outputs/phase3_val_sad.wav
Saved: /Users/andrewchen/Desktop/18789/emonarrify/outputs/phase3_val_surprise.wav


In [ ]:
# ===== Export Deliverables =====
# CRITICAL: Phase 2 depends on this export

lookup_path = LOOKUP_TABLE_PATH
ckpt_path = WEIGHTS_DIR / 'phase3_tts.pt'

# 1) Export lookup table JSON for Phase 2
lookup.export_json(str(lookup_path))

# 2) Save final checkpoint (model + lookup)
torch.save(
    {
        'tts_model': tts_backbone.state_dict(),
        'lookup_table': lookup.state_dict(),
        'sample_rate': SAMPLE_RATE,
        'model_name': 'ConditionalNeuralTTSBackbone',
    },
    ckpt_path,
)

print(f'Lookup table exported to {lookup_path}')
print(f'Phase3 checkpoint saved to {ckpt_path}')